# Deep Learning — Projeto Final
> Previsão de cancelamento de reservas hoteleiras com redes neuronais (FNN e CNN)

**Curso:** Pós-graduação em Data Science e Business Intelligence  
**Disciplina:** Deep Learning | **Aluno:** Ricardo Filipe Fernandes da Silva


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

from src.config       import TARGET, CATEGORICAL_COLS, NUMERIC_COLS, DEFAULT_DATA_PATH
from src.data         import load_data, reconstruct_booking_date
from src.preprocessing import clean_data, split_data, prepare_splits
from src.eda          import plot_missing, plot_target, plot_numeric, plot_categorical
from src.models       import (build_fnn_baseline, build_fnn_dense64_32,
                               build_fnn_dropout, build_cnn_baseline,
                               build_cnn_regularized)
from src.train        import train_model, train_model_no_early_stop
from src.evaluate     import evaluate_model, plot_history, plot_confusion_matrix, compare_models, reshape_for_cnn

print("TensorFlow:", tf.__version__)


## 2. Problema e Objetivo

**Problema:** prever, no momento da reserva, se uma reserva hoteleira será cancelada — classificação binária com dados tabulares.  
**Target:** `is_canceled` (0 = não cancelado · 1 = cancelado)  
**Métrica principal:** accuracy. Na avaliação final: precision, recall e F1-score por classe.  
**Dataset:** [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) — 119 390 reservas, 32 variáveis originais, período 2015–2017.

**Abordagens:**
- **FNN** (Feedforward Neural Network / dense layers) — baseline principal, mais natural para dados tabulares
- **CNN** (Conv1D) — avalia se convolution layers acrescentam valor preditivo


## 3. Load Data

In [ ]:
df = load_data(DEFAULT_DATA_PATH)
display(df.head(3))


## 4. Análise Exploratória (EDA)

EDA focada nas decisões que afectam directamente o modelo: missing values, distribuição da target e variáveis com maior poder discriminativo.


In [ ]:
plot_missing(df)


In [ ]:
plot_target(df)


In [ ]:
# Variáveis com maior diferença entre classes — as mais relevantes para o modelo
for col in ["lead_time", "deposit_type", "market_segment"]:
    if df[col].dtype == object:
        plot_categorical(df, col)
    else:
        plot_numeric(df, col)


**Principais insights:**

| Variável | Observação |
|---|---|
| `lead_time` | Distribuição claramente diferente entre cancelados e não cancelados — feature mais discriminativa |
| `deposit_type` | Categoria *Non Refund* apresenta taxa de cancelamento muito acima da média |
| `market_segment` | Segmento *Groups* com risco de cancelamento superior |
| `company` | 94% de missing — será removida |
| Duplicados | ~32k registos com todas as colunas iguais — mantidos (ver Sec. 5) |
| Target | ~63% não cancelado / ~37% cancelado — sem desequilíbrio extremo |


## 5. Preparação dos Dados

**Decisões metodológicas:**

- **Duplicados mantidos** — sem identificador único de reserva é impossível confirmar que são redundância real; a sua remoção alteraria a distribuição da target de 37% para 27%, o que distorceria o treino.
- **Leakage removido** — excluídas variáveis que não estariam disponíveis no momento da reserva: `reservation_status`, `reservation_status_date`, `booking_changes`, `days_in_waiting_list`, `assigned_room_type`.
- **Split temporal** — divisão cronológica com base em `booking_date` (reconstruída via `arrival_date − lead_time`) para simular um cenário real de previsão: treino → até Jun 2016 · validação → Jul–Dez 2016 · teste → 2017.
- **Preprocessing** — `StandardScaler` para numéricas; `OneHotEncoder` para categóricas; fit exclusivo no treino.


In [ ]:
# Limpeza
df_dated = reconstruct_booking_date(df)
df_clean  = clean_data(df_dated)

# Split temporal
train_df, val_df, test_df = split_data(df_clean)

# Preprocessing (fit apenas no treino)
X_train_p, X_val_p, X_test_p, y_train, y_val, y_test, preprocessor = prepare_splits(
    train_df, val_df, test_df
)

# Reshape para CNN
X_train_cnn = reshape_for_cnn(X_train_p)
X_val_cnn   = reshape_for_cnn(X_val_p)
X_test_cnn  = reshape_for_cnn(X_test_p)

INPUT_DIM     = X_train_p.shape[1]
INPUT_DIM_CNN = X_train_cnn.shape[1]


## 6. Modelação — FNN

Progressão em três passos: baseline mínimo → arquitetura mais complexa (para demonstrar overfitting) → regularização.

| # | Arquitectura | Regularização |
|---|---|---|
| Baseline | `Dense(1, sigmoid)` | — |
| Complexo | `Dense(64) → Dense(32) → Dense(1)` | — |
| **Final ★** | `Dense(64) → Drop(0.5) → Dense(32) → Drop(0.5) → Dense(1)` | Dropout(0.5) + EarlyStopping |

> Foram também testados Dropout(0.3), L2 e batch_size=64. Não melhoraram face à configuração final — omitidos do notebook para clareza.


In [ ]:
# ── FNN Baseline ──────────────────────────────────────────────────────────────
fnn_baseline = build_fnn_baseline(INPUT_DIM)
fnn_baseline.summary()

history_fnn_baseline = train_model_no_early_stop(
    fnn_baseline, X_train_p, y_train, X_val_p, y_val, epochs=20, batch_size=32
)
print("\nFNN Baseline:")
fnn_baseline_results = evaluate_model(fnn_baseline, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
plot_history(history_fnn_baseline, "FNN Baseline")


O baseline mínimo — equivalente a uma regressão logística implementada em Keras — atinge ~86% no treino e ~77% em validação e teste. Serve como ponto de referência: qualquer modelo mais complexo deve superar estes valores para justificar a sua complexidade adicional.


In [ ]:
# ── FNN Complexo — demonstrar overfitting ─────────────────────────────────────
fnn_complex = build_fnn_dense64_32(INPUT_DIM)

history_fnn_complex = train_model_no_early_stop(
    fnn_complex, X_train_p, y_train, X_val_p, y_val, epochs=20, batch_size=32
)
print("\nFNN Complexo:")
fnn_complex_results = evaluate_model(fnn_complex, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
plot_history(history_fnn_complex, "FNN Complexo (sem regularização)")


Com duas camadas ocultas, o treino atinge ~94% de accuracy — mas validação e teste ficam pelos ~77%. A `val_loss` aumenta progressivamente: **overfitting claro**. Este comportamento motiva a regularização.


In [ ]:
# ── FNN Final — Dropout(0.5) + EarlyStopping ─────────────────────────────────
fnn_final = build_fnn_dropout(INPUT_DIM, dropout=0.5)

history_fnn_final = train_model(
    fnn_final, X_train_p, y_train, X_val_p, y_val, epochs=20, batch_size=32, patience=5
)
print("\nFNN Final:")
fnn_final_results = evaluate_model(fnn_final, X_train_p, y_train, X_val_p, y_val, X_test_p, y_test)
plot_history(history_fnn_final, "FNN Final — Dropout(0.5)")


O Dropout(0.5) com EarlyStopping controla o overfitting e melhora a generalização: **~79% de accuracy em validação e ~79% em teste** — o melhor resultado FNN do projecto. As curvas de treino e validação convergem, sem degradação da `val_loss`.


## 7. Modelação — CNN

Os dados tabulares foram reformulados de `(n, features)` para `(n, features, 1)` para compatibilidade com `Conv1D`. Esta adaptação trata cada observação como uma sequência unidimensional de atributos — estrutura não naturalmente convolucional, mas metodologicamente válida para avaliar o potencial das convolution layers neste contexto.

| # | Arquitectura | Regularização |
|---|---|---|
| Baseline | `Conv1D(32) → MaxPool → Flatten → Dense(1)` | — |
| **Final ★** | `Conv1D(32) → Conv1D(64) → MaxPool → Flatten → Dense(32) → Drop(0.5) → Dense(1)` | Dropout(0.5) + EarlyStopping |

> Arquiteturas intermédias (Complex1, Complex2, bs=64) testadas e omitidas — comportamento análogo ao da FNN: overfitting sem regularização, controlado com Dropout.


In [ ]:
# ── CNN Baseline ──────────────────────────────────────────────────────────────
cnn_baseline = build_cnn_baseline(INPUT_DIM_CNN)
cnn_baseline.summary()

history_cnn_baseline = train_model_no_early_stop(
    cnn_baseline, X_train_cnn, y_train, X_val_cnn, y_val, epochs=20, batch_size=32
)
print("\nCNN Baseline:")
cnn_baseline_results = evaluate_model(cnn_baseline, X_train_cnn, y_train, X_val_cnn, y_val, X_test_cnn, y_test)
plot_history(history_cnn_baseline, "CNN Baseline")


O baseline CNN atinge ~88% no treino e ~76–77% em validação e teste — desempenho competitivo face ao baseline FNN, o que confirma que convolution layers conseguem captar padrões relevantes mesmo em dados tabulares.


In [ ]:
# ── CNN Final — regularizada ──────────────────────────────────────────────────
cnn_final = build_cnn_regularized(INPUT_DIM_CNN, dropout=0.5)

history_cnn_final = train_model(
    cnn_final, X_train_cnn, y_train, X_val_cnn, y_val, epochs=20, batch_size=32, patience=5
)
print("\nCNN Final:")
cnn_final_results = evaluate_model(cnn_final, X_train_cnn, y_train, X_val_cnn, y_val, X_test_cnn, y_test)
plot_history(history_cnn_final, "CNN Final — Dropout(0.5)")


Com regularização, a CNN final atinge ~79% em validação e ~78% em teste, com curvas mais estáveis. O desempenho global é ligeiramente inferior ao da FNN final, mas a CNN mostra F1-score mais equilibrado na classe de cancelamentos (ver Sec. 8).


## 8. Comparação Final

In [ ]:
results = {
    "FNN Baseline":   fnn_baseline_results,
    "FNN Complexo":   fnn_complex_results,
    "FNN Final ★":    fnn_final_results,
    "CNN Baseline":   cnn_baseline_results,
    "CNN Final ★":    cnn_final_results,
}
display(compare_models(results))


In [ ]:
# Confusion matrix e classification report dos dois melhores modelos
plot_confusion_matrix(fnn_final, X_test_p,   y_test, title="FNN Final")
plot_confusion_matrix(cnn_final, X_test_cnn, y_test, title="CNN Final")


**Interpretação:**

- A **FNN final** obtém o melhor desempenho global — accuracy de validação ~79,1% e teste ~78,8%.
- A **CNN final** é ligeiramente inferior em accuracy global (~78,5% / ~78,3%), mas apresenta F1-score na classe *Canceled* (1) marginalmente superior, o que indica melhor equilíbrio entre precisão e recall na deteção de cancelamentos — o evento de maior interesse prático.
- Ambos os modelos mostram maior robustez na classe 0 (não cancelado) do que na classe 1 (cancelado). Os **false negatives** — cancelamentos não detectados — são o tipo de erro mais relevante em contexto hoteleiro.
- A FNN é a solução mais natural para dados tabulares independentes. A CNN demonstrou valor analítico e comportamento competitivo, mas sem evidência suficiente de vantagem prática para justificar a sua preferência.


## 9. Utilidade do Modelo

**Aplicações práticas em contexto hoteleiro:**
- Sinalização precoce de reservas com maior risco de cancelamento
- Apoio à gestão de ocupação e planeamento de overbooking
- Acompanhamento diferenciado de perfis de cliente de maior risco

O modelo funciona como **ferramenta de apoio à decisão** — não substitui o conhecimento do negócio, mas reduz incerteza e complementa a análise operacional com informação quantitativa.


In [ ]:
# Exemplo de previsão para uma nova reserva (perfil de risco elevado)
new_booking = pd.DataFrame([{
    "hotel": "City Hotel", "lead_time": 300,
    "arrival_date_year": 2017, "arrival_date_month": "August",
    "arrival_date_week_number": 35, "arrival_date_day_of_month": 25,
    "stays_in_weekend_nights": 1, "stays_in_week_nights": 3,
    "adults": 2, "children": 0, "babies": 0, "meal": "BB",
    "country": "PRT", "market_segment": "Online TA",
    "distribution_channel": "TA/TO", "is_repeated_guest": 0,
    "previous_cancellations": 5, "previous_bookings_not_canceled": 0,
    "reserved_room_type": "A", "deposit_type": "Non Refund",
    "agent": "9.0", "customer_type": "Transient",
    "adr": 180.0, "required_car_parking_spaces": 0,
    "total_of_special_requests": 0
}])

for col in CATEGORICAL_COLS:
    new_booking[col] = new_booking[col].astype(str)

new_p    = preprocessor.transform(new_booking).toarray()
pred_prob = fnn_final.predict(new_p, verbose=0)[0][0]
print(f"Probabilidade de cancelamento : {pred_prob:.4f}")
print(f"Classe prevista               : {'Cancelado' if pred_prob >= 0.5 else 'Não cancelado'}")


## 10. Conclusões, Limitações e Trabalho Futuro

**Conclusões:**
- O baseline mínimo (~77% teste) demonstra que o dataset tem sinal preditivo robusto.
- O overfitting é o principal desafio: arquiteturas mais complexas sem regularização não generalizam.
- **Dropout(0.5) + EarlyStopping** foi a estratégia de regularização mais eficaz em ambas as famílias.
- A **FNN** é a solução mais eficaz para este problema tabular. A **CNN** demonstrou comportamento competitivo e F1-score ligeiramente superior na classe de cancelamentos, sem evidência clara de vantagem global.
- A divisão temporal e a exclusão de variáveis com leakage foram decisões metodológicas críticas para uma avaliação realista.

**Limitações:**
- O modelo depende da representatividade dos dados históricos — pode degradar em outros contextos hoteleiros ou períodos.
- O threshold de decisão (0.5) não foi optimizado para o custo específico de cada tipo de erro.
- A abordagem CNN usa Conv1D em dados tabulares, estrutura não naturalmente convolucional.

**Trabalho futuro:**
- Optimizar o decision threshold em função do custo de negócio (false negatives vs. false positives)
- Explorar embeddings para variáveis categóricas com muitas categorias (`country`, `agent`)
- Aplicar SHAP para explicabilidade e identificação das features mais relevantes
- Validar a solução noutros contextos hoteleiros e períodos temporais
